In [1]:
import pandas as pd
from pathlib import Path
from algo.features import add_labels
from algo.backtester import backtest_ML_switchable, hybrid_entry, default_exit
from algo.model import load_or_train, predict_last    # or your correct import path
from algo.features import add_indicators
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names"
)
# if needed

# === Step 1: Load CSV and set 'date' as datetime index ===
csv_path = Path("data_processed/HDFCBANK_5minute_feat.csv")
df = pd.read_csv(csv_path, parse_dates=["date"], index_col="date")
df = add_labels(df)
df = add_indicators(df)  # <--- All features added here
split_date = '2024-01-01'
df_train = df[df.index < split_date].copy()
df_test  = df[df.index >= split_date].copy()



# 2. Train or load your ML model (adjust path/horizon if needed)
model = load_or_train(df_train, retrain=False, horizon=5)

# 3. Run the backtest
trades, metrics = backtest_ML_switchable(
    df=df_test,
    model=model,
    predict_fn=predict_last,
    entry_rule_fn=hybrid_entry,   # <--- put your rule here!
    exit_rule_fn=default_exit,    # or None to fallback to basic SL/TP/EOD
    capital=100_000,
    contract_size=10,
    lookback=30,      # must match model lookback!
    sl_pct=0.0015,      # 1% stop
    tp_pct=0.0040,      # 2% target
    debug=False
)

# 4. Inspect results:
print(metrics)
print(trades.head())


Prepared 5746 samples | Class balance (mean): 0.515
🔧  Training started …
✅  Finished in 0.6s   (best_iter = 5, best_AUC = 0.5529)
Hold-out accuracy: 0.538
Non-NaN ml_prob: 28362 out of 28392
            ml_prob
count  28362.000000
mean       0.508561
std        0.006711
min        0.498664
25%        0.504045
50%        0.505543
75%        0.512621
max        0.529864
{'Trades': 676, 'WinRate': np.float64(0.28106508875739644), 'GrossPnL': np.float64(2323.1835000003434), 'Fees': np.float64(19188.07402547197), 'NetPnL': np.float64(-16864.89052547163), 'EquityFinal': np.float64(83135.10947452832)}
             entry_ts             exit_ts side  entry_price   exit_price  \
0 2024-01-01 13:10:00 2024-01-01 15:05:00  BUY      1701.30  1698.748050   
1 2024-01-02 12:55:00 2024-01-02 13:35:00  BUY      1698.55  1696.002175   
2 2024-01-02 15:00:00 2024-01-02 15:05:00  BUY      1702.25  1699.696625   
3 2024-01-04 13:10:00 2024-01-04 13:15:00  BUY      1689.10  1686.566350   
4 2024-01-04 13:2

In [2]:
df

,open,high,low,close,volume,fees_estimate,atr,atr_median20,volatility_5,volatility_10,...,is_hammer,is_bullish_engulfing,body_range_ratio,future_return,volatility,price_vs_vwap,vwap_gap,trend_strength,bb_position,label
date,,,,,,,,,,,,,,,,,,,,,
2022-01-03 09:15:00,1485.00,1489.35,1480.50,1488.95,163590,26.409506,NaN,NaN,NaN,NaN,...,0,0,0.446328,0.000672,NaN,2.683333,2.683333,NaN,NaN,0
2022-01-03 09:20:00,1488.90,1489.00,1482.35,1483.25,83226,26.308405,NaN,NaN,NaN,NaN,...,0,0,0.849624,0.007214,NaN,-2.544589,-2.544589,NaN,NaN,0
2022-01-03 09:25:00,1483.25,1486.75,1483.05,1485.60,46758,26.350087,NaN,NaN,NaN,NaN,...,0,0,0.635135,0.007808,NaN,-0.089269,-0.089269,NaN,NaN,0
2022-01-03 09:30:00,1485.75,1487.00,1483.40,1487.00,60896,26.374919,NaN,NaN,NaN,NaN,...,0,0,0.347222,0.009280,NaN,1.291708,1.291708,NaN,NaN,0
2022-01-03 09:35:00,1487.00,1488.95,1486.20,1488.50,47063,26.401524,NaN,NaN,NaN,NaN,...,0,0,0.545455,0.010178,NaN,2.536775,2.536775,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-11 15:05:00,1984.80,1984.90,1983.10,1983.60,276723,35.183113,1.920150,2.001635,0.000355,0.000333,...,0,0,0.666667,NaN,0.000404,-4.767106,-4.767106,0.031011,0.220923,0
2025-07-11 15:10:00,1983.30,1984.00,1982.50,1982.80,309306,35.168924,1.878135,2.001635,0.000392,0.000334,...,0,0,0.333333,NaN,0.000414,-5.253300,-5.253300,0.049926,0.051142,0
2025-07-11 15:15:00,1982.70,1984.90,1982.50,1983.30,319388,35.177792,1.930321,1.977257,0.000421,0.000334,...,0,0,0.250000,NaN,0.000414,-4.493277,-4.493277,0.059214,0.157255,0


In [1]:
import pandas as pd
from algo.features import add_labels
# load
df = pd.read_csv("data_processed/HDFCBANK_5minute_feat.csv", parse_dates=["date"], index_col="date")
df = add_labels(df, horizon=5)




In [2]:
print(df["label"].value_counts(normalize=True))
# Ideally you want something like 60:40, not 95:5


label
0    0.921321
1    0.078679
Name: proportion, dtype: float64


In [3]:
for col in df.columns:
    if col not in ["label", "date"] and pd.api.types.is_numeric_dtype(df[col]):
        means = df.groupby("label")[col].mean()
        print(f"{col:20s}  mean(0)={means.get(0, float('nan')):.4f}  mean(1)={means.get(1, float('nan')):.4f}")


open                  mean(0)=1602.6077  mean(1)=1568.0766
high                  mean(0)=1603.9424  mean(1)=1569.8834
low                   mean(0)=1601.2181  mean(1)=1566.2665
close                 mean(0)=1602.6140  mean(1)=1568.0969
volume                mean(0)=178678.2629  mean(1)=297398.1191
atr                   mean(0)=2.8050  mean(1)=3.3940
minute_of_day         mean(0)=737.9146  mean(1)=763.7697
ret1                  mean(0)=0.0000  mean(1)=-0.0000
ret5                  mean(0)=0.0000  mean(1)=0.0002
body_1                mean(0)=0.4510  mean(1)=0.5104
below_low_10          mean(0)=0.0672  mean(1)=0.0827
vwap                  mean(0)=1602.5663  mean(1)=1567.8988
close_vs_vwap         mean(0)=0.0000  mean(1)=0.0001
ema_8                 mean(0)=1602.6132  mean(1)=1567.9245
ema_21                mean(0)=1602.5959  mean(1)=1567.7736
rsi_14                mean(0)=50.3588  mean(1)=51.1581
macd                  mean(0)=0.0487  mean(1)=0.0970
macd_signal           mean(0)=0.0542  me

In [15]:
print(df.corr()["label"].sort_values(ascending=False))


label             1.000000
future_return     0.546720
atr               0.139421
range_1           0.133627
volatility        0.095824
trend_strength    0.081337
vol_spike         0.072514
minute_of_day     0.064310
volume            0.059909
bb_upper_touch    0.040054
body_1            0.039758
bb_lower_touch    0.024734
above_high_10     0.024205
below_low_10      0.016513
bb_position       0.013005
rsi_14            0.012755
ret5              0.011925
up_streak         0.009510
vwap_gap          0.006467
price_vs_vwap     0.006467
close_vs_vwap     0.006368
macd              0.004770
ret1             -0.002357
macd_signal      -0.002370
down_streak      -0.004275
bb_upper         -0.058606
high             -0.059811
close            -0.060631
open             -0.060652
vwap             -0.060827
ema_8            -0.060957
ema_21           -0.061236
low              -0.061399
bb_lower         -0.064132
Name: label, dtype: float64


In [2]:
from algo.model import load_or_train
model = load_or_train(df, retrain=True, horizon=5)

Prepared 9920 samples | Class balance (mean): 0.515
🔧  Training started …
✅  Finished in 1.6s   (best_iter = 167, best_AUC = 0.5453)
Hold-out accuracy: 0.518


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
lgb_model = model.named_steps['lgb']
  # No .named_steps here!

feature_names = [f"f{i}" for i in range(lgb_model.n_features_in_)]
importances = lgb_model.feature_importances_

import pandas as pd
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(importance_df.head(20))


    feature  importance
709    f709           7
716    f716           6
562    f562           5
465    f465           4
293    f293           4
205    f205           3
136    f136           3
172    f172           2
571    f571           2
702    f702           2
60      f60           2
585    f585           2
393    f393           2
269    f269           2
636    f636           1
628    f628           1
682    f682           1
304    f304           1
438    f438           1
597    f597           1


In [7]:
LOOKBACK = 30
from algo.features import FEATURES
print(f"LOOKBACK: {LOOKBACK}, len(FEATURES): {len(FEATURES)}")
print(FEATURES)


LOOKBACK: 30, len(FEATURES): 24
['ret1', 'ret5', 'atr', 'ema_8', 'ema_21', 'vwap', 'close_vs_vwap', 'body_1', 'rsi_14', 'macd', 'macd_signal', 'below_low_10', 'bb_upper', 'bb_lower', 'bb_upper_touch', 'bb_lower_touch', 'volatility_5', 'volatility_10', 'vol_spike', 'minute_of_day', 'price_vs_vwap', 'trend_strength', 'vwap_gap', 'bb_position']


In [8]:
num_feats = len(FEATURES)
lookback = LOOKBACK  # make sure this is 30, or whatever you use

def decode_fidx(idx, num_feats=num_feats, lookback=lookback):
    lookback_idx = idx // num_feats
    feat_idx = idx % num_feats
    t_idx = lookback - lookback_idx - 1  # t-1 is most recent, t-2 next, ...
    return f"t-{t_idx} {FEATURES[feat_idx]}"

for idx in importance_df['feature']:
    fidx = int(str(idx).replace('f',''))
    print(f"Feature: {decode_fidx(fidx)}, Importance: {importance_df.loc[importance_df['feature'] == idx, 'importance'].values[0]}")


Feature: t-0 bb_lower, Importance: 7
Feature: t-0 price_vs_vwap, Importance: 6
Feature: t-6 macd_signal, Importance: 5
Feature: t-10 macd, Importance: 4
Feature: t-17 vwap, Importance: 4
Feature: t-21 bb_lower, Importance: 3
Feature: t-24 volatility_5, Importance: 3
Feature: t-22 ema_21, Importance: 2
Feature: t-6 minute_of_day, Importance: 2
Feature: t-0 close_vs_vwap, Importance: 2
Feature: t-27 bb_upper, Importance: 2
Feature: t-5 macd, Importance: 2
Feature: t-13 macd, Importance: 2
Feature: t-18 vwap, Importance: 2
Feature: t-3 bb_upper, Importance: 1
Feature: t-3 ema_21, Importance: 1
Feature: t-1 macd_signal, Importance: 1
Feature: t-17 volatility_5, Importance: 1
Feature: t-11 close_vs_vwap, Importance: 1
Feature: t-5 trend_strength, Importance: 1
Feature: t-2 trend_strength, Importance: 1
Feature: t-11 macd, Importance: 1
Feature: t-26 vwap, Importance: 1
Feature: t-15 macd_signal, Importance: 1
Feature: t-20 ema_21, Importance: 1
Feature: t-22 macd_signal, Importance: 1
Featu

In [18]:
import pandas as pd
import numpy as np

# --- 1. Set your LOOKBACK and FEATURES exactly as used for training ---
LOOKBACK = 30  # Make sure this matches your model.py!
FEATURES = [
    "ema_21", "bb_lower_touch", "vol_spike", "macd_signal", "bb_lower",
    "below_low_10", "ret1", "vwap", "above_high_10", "bb_lower", "macd",
    "bb_upper", "minute_of_day", "ret5", "ret5", "up_streak", "bb_lower_touch",
    "body_1", "close_vs_vwap", "vwap", "ret5", "macd_signal", "body_1", "close_vs_vwap",
    "vwap", "ret5"
    # ... fill in your full FEATURES list here exactly as used in training!
]
# NOTE: The FEATURES list must match what was used in model training.

# --- 2. Get feature importances from the model ---
lgb_model = model  # or model.named_steps['lgb'] if using pipeline
importances = lgb_model.feature_importances_

# --- 3. Expand features for each lag ---
expanded_features = []
for lag in range(-LOOKBACK, 0):  # Lags: t-30, t-29, ..., t-1, t-0 if LOOKBACK=30
    for feat in FEATURES:
        expanded_features.append(f"t{lag} {feat}")

assert len(expanded_features) == len(importances), (
    f"Mismatch: {len(expanded_features)} features vs {len(importances)} importances"
)

# --- 4. Create DataFrame ---
feature_importances = pd.DataFrame({
    'feature': expanded_features,
    'importance': importances
})

# --- 5. Extract indicator/feature name from the lagged feature string ---
def extract_indicator(feature_name):
    # Example: "t-5 ema_21" -> "ema_21"
    return feature_name.split(" ", 1)[1]

feature_importances['indicator'] = feature_importances['feature'].apply(extract_indicator)
feature_importances['lag'] = feature_importances['feature'].str.extract(r't(-?\d+)').astype(int)

# --- 6. Filter for features with importance >= 10 ---
filtered = feature_importances[feature_importances['importance'] >= 10]

# --- 7. Summarize: group by indicator and show total importance & top lags ---
summary = (
    filtered.groupby('indicator')
    .agg(
        total_importance=('importance', 'sum'),
        top_lags=('lag', lambda lags: list(sorted(lags)))
    )
    .sort_values('total_importance', ascending=False)
)
print(summary)

# Optionally, print the list of features to KEEP (for re-training):
keep_features = filtered['feature'].tolist()
print("Important features to KEEP (for re-training):")
for feat in keep_features:
    print(feat)


                total_importance  \
indicator                          
ret5                         528   
close_vs_vwap                333   
bb_lower                     197   
body_1                       187   
macd_signal                  147   
minute_of_day                139   
vwap                         126   
macd                         106   
above_high_10                103   
ema_21                        73   
bb_lower_touch                59   
below_low_10                  55   
ret1                          35   

                                                         top_lags  
indicator                                                          
ret5            [-29, -28, -25, -25, -24, -23, -22, -21, -21, ...  
close_vs_vwap   [-30, -29, -25, -25, -24, -22, -21, -20, -19, ...  
bb_lower        [-30, -29, -29, -26, -21, -18, -9, -8, -7, -6,...  
body_1                        [-25, -22, -20, -9, -2, -2, -1, -1]  
macd_signal     [-27, -26, -24, -23, -21, -14, -13,

DATASET LOADING AND TRAINING

In [16]:
import pandas as pd
from pathlib import Path
from features import add_indicators
# Update filename as needed
csv_path = Path("data/HDFCBANK_5minute.csv")
df_raw = pd.read_csv(csv_path, index_col=0, parse_dates=True).sort_index()
print(f"Loaded {df_raw.shape[0]} rows")


df_feat = add_indicators(df_raw)
print("Indicators added:", list(df_feat.columns))


Loaded 65291 rows
Indicators added: ['open', 'high', 'low', 'close', 'volume', 'atr', 'volatility_5', 'volatility_10', 'minute_of_day', 'ret1', 'ret5', 'body_1', 'below_low_10', 'vwap', 'close_vs_vwap', 'vol_spike', 'ema_8', 'ema_21', 'rsi_14', 'macd', 'macd_signal', 'bb_upper', 'bb_lower', 'bb_upper_touch', 'bb_lower_touch']


In [17]:
from features import add_labels

df_feat = add_labels(df_feat)
print("Labels added. Sample:")
print(df_feat[["label"]].value_counts())


Labels added. Sample:
label
0        60154
1         5137
Name: count, dtype: int64


In [24]:
from model import load_or_train
model = load_or_train(df_feat,retrain=True,horizon=5)


Prepared 9920 samples | Class balance (mean): 0.515
🔧  Training started …
✅  Finished in 0.6s   (best_iter = 7, best_AUC = 0.5347)
Hold-out accuracy: 0.524


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [30]:
import pandas as pd

# Assume: importance_df has columns ['feature', 'importance'], with 'feature' like 'f0', 'f1', ...
# Assume: X is your lagged feature DataFrame with columns ['t-0 macd', ...]
# Assume: FEATURES is your enabled indicators list
# Assume: LOOKBACK is your lookback window (usually 30)

num_feats = len(FEATURES)
lookback = LOOKBACK

def decode_fidx(idx, num_feats=num_feats, lookback=lookback):
    lookback_idx = idx // num_feats
    feat_idx = idx % num_feats
    t_idx = lookback - lookback_idx - 1  # so t-0 is most recent
    return f"t-{t_idx} {FEATURES[feat_idx]}"

# Add decoded names
importance_df['decoded_name'] = importance_df['feature'].apply(
    lambda x: decode_fidx(int(str(x).replace('f','')))
)

# See your top features with decoded names:
print(importance_df[['decoded_name', 'importance']].sort_values('importance', ascending=False).head(20))


           decoded_name  importance
709        t-0 bb_lower           7
716   t-0 price_vs_vwap           6
562     t-6 macd_signal           5
293           t-17 vwap           4
465           t-10 macd           4
205       t-21 bb_lower           3
136   t-24 volatility_5           3
60        t-27 bb_upper           2
269           t-18 vwap           2
393           t-13 macd           2
585            t-5 macd           2
702   t-0 close_vs_vwap           2
571   t-6 minute_of_day           2
172         t-22 ema_21           2
704          t-0 rsi_14           1
281  t-18 volatility_10           1
13        t-29 bb_lower           1
201           t-21 macd           1
197           t-21 vwap           1
706     t-0 macd_signal           1


In [35]:
importance_df['decoded_name'] = importance_df['feature'].apply(
    lambda x: decode_fidx(int(str(x).replace('f','')))
)


In [41]:
# --- Prune helper ---
def make_pruned_df(df_full, importance_df, label_col='label', importance_threshold=0):
    keep_cols = importance_df[importance_df.importance > importance_threshold]['decoded_name'].tolist()
    keep_cols = [c for c in keep_cols if c in df_full.columns]
    return df_full[keep_cols + [label_col]].copy()

# --- Prune and retrain ---
df_pruned = make_pruned_df(df_feat, importance_df, label_col='label', importance_threshold=0)

from model import load_or_train
model = load_or_train(df_pruned, retrain=True)


KeyError: 'close'